# Statistical analyses between Indoor Environmental Quality factors and students' behavioral and emotional varaibles (engagement, attention, and interaction)

> Students' variables are estimated using Valdes-Ramirez et al. (2026) and Valdes-Ramirez et al. (2023)
>
> Indoor envirinmental Quality factors are measured using Valdes-Ramirez et al. (2025)
>
> Dataset and source code are freely available and can be used for research purposes.

1. [Loading dataset and libraries](#1)
2. [Preprocessing data](#2)
3. [Time series and violin plots](#3)
4. [Correlation analysis](#4)
5. [Cross-Correlation Function analysis](#5)
6. [Time-Varying Granger Causality tests](#6)
7. [Independence analysis with categorized variables](#7)

<a id='1'></a> 
## 1. Loading dataset and libraries 

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from scipy import stats
from scipy.signal import find_peaks
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import grangercausalitytests, acf, pacf, adfuller
from statsmodels.tsa.ar_model import AutoReg
from scipy.stats import spearmanr, chi2_contingency
import matplotlib.patches as mpatches
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import plotly.io as pio
import matplotlib.pyplot as plt
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load the CSV file
file_path = 'iclassroom_indicator_sensors_op1.csv'  # Replace with your actual file path
raw_data = pd.read_csv(file_path)
# Change AQI type to category
raw_data['AQI'] = raw_data['AQI'].astype('category')
data = raw_data.copy()
# Transform the column 'oxygenVol' to two decimal places
data = data.round(2)
#data.info()
info=data.info()

In [ ]:
studied_vars = ["Temperature", "Humidity", "eCO2", "AQI", "TVOC", "LUX", "oxygenVol", "Engagement", 'Attention', 'Interaction']
ieq_vars = ["Temperature", "Humidity", "eCO2", "AQI", "TVOC", "LUX", "oxygenVol"]
cleaned_ieq_vars = ["Temperature", "Humidity", "TVOC", "LUX", "oxygenVol"]
student_vars = ["Engagement", 'Attention', 'Interaction']

<a id='2'></a> 
## 2. Preprocessing data 

In [ ]:
# Combine the year, month, day, hour, minute, second columns into a single datetime column
data['Date'] = pd.to_datetime(data[['year', 'month', 'day', 'hour', 'minute', 'seconds']])
# There are some timestamps going back in the time series, we must fix them by sorting the data by date
data = data.sort_values(by='Date')

data = data[studied_vars+['Date']].dropna(subset=studied_vars)
# Set the datetime as the index
#data.set_index('Date', inplace=True)
data.head()


In [ ]:
# Check types of each column
data_types = data.dtypes
print(data_types)

In [ ]:
# Create 3-category splits for each environmental variable (except AQI which is binary: 0->low, 1->high)
df_cat = data.copy()

for var in ieq_vars + student_vars:
    cat_col = f"{var}_cat"
    if var == "AQI":
        # AQI: low if 0, high if 1 (others become NaN)
        df_cat[cat_col] = df_cat[var].map({1: "low", 2: "high"}).astype("category")
    else:
        # split into low/mid/high using 33% and 66% quantiles
        q33, q66 = df_cat[var].quantile([0.33, 0.66]).values
        df_cat[cat_col] = pd.cut(df_cat[var], bins=[-np.inf, q33, q66, np.inf], labels=["low", "mid", "high"])

# Subsampling df_cat every 60 rows (5min = 60 samples) to avoid data dependency for Chi-square tests
df_cat_5min = df_cat.iloc[::60, :].reset_index(drop=True) 


In [ ]:
# Prepare 5-minute resampled dataset (mean aggregation)
# Drop AQI column as it is binary and not suitable for Spearman correlation
RESAMPLE_RULE = "5min"
data_5min = data.drop(columns=['AQI'], inplace=False)
data_5min = data_5min.set_index("Date").sort_index().resample(RESAMPLE_RULE).mean().dropna()


In [ ]:
data_5min_AQI = data.set_index("Date").sort_index().resample(RESAMPLE_RULE).first()['AQI'].dropna()


In [ ]:
data_5min_AQI = pd.concat([data_5min_AQI, data_5min[['Engagement', 
                                                     'Attention', 
                                                     'Interaction']]], axis=1)
data_5min_AQI

<a id='3'></a>
## Time series and Violin plots

In [ ]:
## Plotting behavioral variables over time

# Define the variables you want to plot
plot_columns = data[["Date", "Engagement", "Attention", "Interaction"]]

# Create subplots: 7 rows, 1 column
fig = make_subplots(
    rows=3, cols=1,  # 7 rows, 1 column for the plots
    subplot_titles=plot_columns.columns[1:],  # Titles for each subplot (excluding 'Date')
    shared_xaxes=True,  # Share the x-axis across all plots (Date)
    vertical_spacing=0.09  # Spacing between plots
)

# Define custom Y-axis titles for each variable
y_axis_titles = ["%", "%", "%"]

# Plot each variable in a different row
variables = plot_columns.columns[1:]  # Exclude 'Date' column
for i, var in enumerate(variables):
    fig.add_trace(
        go.Scatter(x=plot_columns['Date'], y=plot_columns[var], mode='lines', name=var),
        row=i+1, col=1  # Plot in the ith row and 1st column
    )
    # Add Y-axis titles
    fig.update_yaxes(title_text=y_axis_titles[i], row=i+1, col=1)

# Update layout for better appearance and font settings
fig.update_layout(
    height=1000,  # Adjust height for all rows
    showlegend=False,  # Hide legend since each plot is titled
    title_text="Students' Response Over Time",
    title_font=dict(size=24, color='black', family='Arial Bold'),  # Increase title font size and use a bold font
    xaxis_title="Date",
    xaxis_title_font=dict(size=24, color='black', family='Arial'),  # X-axis font size and bold font
    yaxis_title_font=dict(size=24, color='black', family='Arial'),  # Y-axis font size and bold font
    font=dict(size=22)  # General font size for other elements
)

# Adjust font size and boldness for subplot titles
for i in range(1, 4):  # Adjust according to the number of subplots (3 in this case)
    fig.update_annotations(font=dict(size=24, family='Arial'))

fig.show()
fig.write_image("students_response_over_time.pdf")


### Plotting time series of student behavior variables against IEQ variables

In [ ]:
# Define axis titles
y_axis_titles_left = [
    "Temperature (°C)", "Humidity (%)", "eCO2 (ppm)", "AQI", 
    "TVOC (ppb)", "Light (Lux)", "Oxygen (Vol)"
]
y_axis_titles_right = ["Engagement (%)"] * 7

# Create subplots with clear separation, better font, and visible legends
fig = make_subplots(
    rows=7, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.01,
    specs=[[{"secondary_y": True}] for _ in range(7)]
)

for i, env_var in enumerate(ieq_vars):
    # Environmental variable (left Y)
    fig.add_trace(
        go.Scatter(
            x=data['Date'],
            y=data[env_var],
            mode='lines',
            name=env_var,
            line=dict(width=2),
        ),
        row=i+1, col=1, secondary_y=False
    )
    # Engagement (right Y)
    fig.add_trace(
        go.Scatter(
            x=data['Date'],
            y=data["Engagement"],
            mode='lines',
            name='Engagement',
            line=dict(color='black', width=2), #dash='dash')
        ),
        row=i+1, col=1, secondary_y=True
    )
    fig.update_yaxes(title_text=y_axis_titles_left[i], row=i+1, col=1, secondary_y=False)
    fig.update_yaxes(title_text=y_axis_titles_right[i], row=i+1, col=1, secondary_y=True)

# Layout improvements
fig.update_layout(
    height=1000,  # Much taller so plots are not cramped
    width=950,
    title_font=dict(size=22, color='black', family='Serif'),
    font=dict(size=16, family='Serif'),
    showlegend=False,
    margin=dict(l=5, r=5, t=5, b=5)
)

# Apply 10-minute step to x-axis
fig.update_xaxes(dtick=600000, tickformat="%H:%M")

# Display (for notebook)
fig.show()

# Save as PDF with good resolution. Requires Kaleido installed.
pio.write_image(fig, "engagement_vs_environmental_variables.pdf", format="pdf", width=950, height=1000, scale=1)


In [ ]:
# Define axis titles
y_axis_titles_left = [
    "Temperature (°C)", "Humidity (%)", "eCO2 (ppm)", "AQI", 
    "TVOC (ppb)", "Light (Lux)", "Oxygen (Vol)"
]
y_axis_titles_right = ["Attention (%)"] * 7

# Create subplots with clear separation, better font, and visible legends
fig = make_subplots(
    rows=7, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.01,
    specs=[[{"secondary_y": True}] for _ in range(7)]
)

for i, env_var in enumerate(ieq_vars):
    # Environmental variable (left Y)
    fig.add_trace(
        go.Scatter(
            x=data['Date'],
            y=data[env_var],
            mode='lines',
            name=env_var,
            line=dict(width=2),
        ),
        row=i+1, col=1, secondary_y=False
    )
    # Engagement (right Y)
    fig.add_trace(
        go.Scatter(
            x=data['Date'],
            y=data["Attention"],
            mode='lines',
            name='Attention',
            line=dict(color='black', width=2), #dash='dash')
        ),
        row=i+1, col=1, secondary_y=True
    )
    fig.update_yaxes(title_text=y_axis_titles_left[i], row=i+1, col=1, secondary_y=False)
    fig.update_yaxes(title_text=y_axis_titles_right[i], row=i+1, col=1, secondary_y=True)

# Layout improvements
fig.update_layout(
    height=1000,  # Much taller so plots are not cramped
    width=950,
    title_font=dict(size=22, color='black', family='Serif'),
    font=dict(size=16, family='Serif'),
    showlegend=False,
    margin=dict(l=5, r=5, t=5, b=5)
)

# Apply 10-minute step to x-axis
fig.update_xaxes(dtick=600000, tickformat="%H:%M")

fig.show()

pio.write_image(fig, "attention_vs_environmental_variables.pdf", format="pdf", width=950, height=1000, scale=1)


In [ ]:
# Define axis titles
y_axis_titles_left = [
    "Temperature (°C)", "Humidity (%)", "eCO2 (ppm)", "AQI", 
    "TVOC (ppb)", "Light (Lux)", "Oxygen (Vol)"
]
y_axis_titles_right = ["Interaction (%)"] * 7

# Create subplots with clear separation, better font, and visible legends
fig = make_subplots(
    rows=7, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.01,
    specs=[[{"secondary_y": True}] for _ in range(7)]
)

for i, env_var in enumerate(ieq_vars):
    # Environmental variable (left Y)
    fig.add_trace(
        go.Scatter(
            x=data['Date'],
            y=data[env_var],
            mode='lines',
            name=env_var,
            line=dict(width=2),
        ),
        row=i+1, col=1, secondary_y=False
    )
    # Engagement (right Y)
    fig.add_trace(
        go.Scatter(
            x=data['Date'],
            y=data["Interaction"],
            mode='lines',
            name='Interaction',
            line=dict(color='black', width=2),
        ),
        row=i+1, col=1, secondary_y=True
    )
    fig.update_yaxes(title_text=y_axis_titles_left[i], row=i+1, col=1, secondary_y=False)
    fig.update_yaxes(title_text=y_axis_titles_right[i], row=i+1, col=1, secondary_y=True)

# Layout improvements
fig.update_layout(
    height=1000,  # Much taller so plots are not cramped
    width=950,
    title_font=dict(size=22, color='black', family='Serif'),
    font=dict(size=16, family='Serif'),
    showlegend=False,
    margin=dict(l=5, r=5, t=5, b=5)
)

# Apply 10-minute step to x-axis
fig.update_xaxes(dtick=600000, tickformat="%H:%M")

# Display (for notebook)
fig.show()

pio.write_image(fig, "interaction_vs_environmental_variables.pdf", format="pdf", width=950, height=1000, scale=1)

### Creating the violin plots 

> Create a graph with 21 violin plots, one for each combination of IEQ variable and sudents behavior variable. The plot has three columns and 7 rows.

In [ ]:
# Prepare subplot grid: 5 rows (IEQ vars) x 3 cols (Engagement, Attention, Interaction)
titles = [f"{env} — {beh}" for env in cleaned_ieq_vars for beh in student_vars]

fig = make_subplots(
    rows=len(cleaned_ieq_vars),
    cols=len(student_vars),
    subplot_titles=titles,
    horizontal_spacing=0.06,
    vertical_spacing=0.05
)

# Colors for categories
colors = {"low": "#1f77b4", "mid": "#ff7f0e", "high": "#064b06"}

# Add violin traces: for each env var (row) and each behaviour (col) add one violin per category
for i, env in enumerate(cleaned_ieq_vars):
    cat_col = f"{env}_cat"
    for j, beh in enumerate(student_vars):
        row, col = i + 1, j + 1
        # show legend only once (first subplot) to avoid duplicate legend entries
        legend_shown = False
        for k, cat in enumerate(["low", "mid", "high"]):
            # For AQI 'mid' may not exist; skip empty categories
            if cat not in df_cat[cat_col].cat.categories:
                continue
            sel = df_cat[df_cat[cat_col] == cat][beh].dropna()
            if sel.empty:
                continue
            # x must be repeated category labels so violins are grouped by category label within the subplot
            x = [cat] * sel.shape[0]
            showlegend = (not legend_shown)  # show legend for the first added category only
            # legend_shown = legend_shown or showlegend
            fig.add_trace(
                go.Violin(
                    x=x,
                    y=sel,
                    name=cat,
                    legendgroup=cat,
                    scalegroup=f"{env}_{beh}",
                    line_color="black",
                    fillcolor=colors.get(cat, "#888"),
                    opacity=0.6,
                    meanline_visible=True,
                    showlegend=showlegend
                ),
                row=row, col=col
            )
        # Y-axis title for the behavior column
        fig.update_yaxes(title_text=f"{beh} (%)", row=row, col=col)

# Increase subplot title font size
for ann in fig['layout']['annotations']:
    ann['font'] = dict(size=22, family="Serif")

# Layout tweaks
fig.update_layout(
    height=1600,
    width=1100,
    template="simple_white",
    boxmode="group",
    title_font=dict(size=22, family='Serif'), # color='black', 
    font=dict(size=22, family='Serif'),
    showlegend=False,
    violingap=0.2,
    violingroupgap=0.1,
    margin=dict(l=5, r=5, t=22, b=13)
)

fig.update_yaxes(title_standoff=2)

# Show and save (requires kaleido for write_image)
fig.show()
pio.write_image(fig, "violin_21_plots.pdf", format="pdf", width=1100, height=1600)

### Descriptive statistics and tests

In [ ]:
description = data[studied_vars].describe().round(2)
description

In [ ]:
# Compute Augmented Dickey-Fuller test for each studied variable
adf_results = {}
for var in set(studied_vars) - {'AQI'}:
    result = adfuller(data[var].dropna())
    adf_results[var] = {
        'ADF Statistic': result[0],
        'p-value': result[1],
        'Used Lag': result[2],
        # 'Number of Observations': result[3],
        # 'Critical Values': result[4]
    }
# round values to 2 decimals
for var in adf_results:
    adf_results[var]['ADF Statistic'] = round(adf_results[var]['ADF Statistic'], 2)
    adf_results[var]['p-value'] = round(adf_results[var]['p-value'], 4)
    adf_results[var]['Used Lag'] = adf_results[var]['Used Lag']
adf_results_df = pd.DataFrame(adf_results).T
print(adf_results_df)

In [ ]:
# Describe the category variables
print(data['AQI'].value_counts())
print(data['AQI'].mode())


In [ ]:
# Checking the var LUX before the artificial light was turned on
data[studied_vars].where(data['Date']<='2024-09-30 18:40:00')['LUX'].describe()

<a id='4'></a>
## 4. Correlational analyses

In [ ]:
# Autocorrelation values for IEQ variables with lag up to 60 = 5 minutes
acf_values = {}
for var in studied_vars:
    acf_values[var] = acf(data[var].dropna(), nlags=60) 
    # round to 2 decimals
    acf_values[var] = [round(x, 2) for x in acf_values[var]]
acf_values_df = pd.DataFrame(acf_values)
# Save to LaTeX file rows 1,2,3,15,30,60
acf_values_df.iloc[[1,2,3,6,12,24,36,60]].to_latex('acf_ieq_values.tex')
acf_values_df.iloc[[1,2,3,6,12,24,36,60]]

### Computing correlation coefficients between IEQ variables

In [ ]:
# Correlation analysis only between IEQ variables using Spearman's rho and Kendall's tau 
# with the subsampled dataset every 5 minuties
correlation_results = []
for env_var in (set(ieq_vars)-{'AQI'}):
        for env_var2 in (set(ieq_vars)-{'AQI'}):
        # Spearman's rho
            spearman_corr, spearman_p = stats.spearmanr(data_5min[env_var], data_5min[env_var2], nan_policy='omit')
            # Kendall's tau
            kendall_corr, kendall_p = stats.kendalltau(data_5min[env_var], data_5min[env_var2], nan_policy='omit')
            correlation_results.append({
                'Environmental Variable': env_var,
                'Environmental Variable2': env_var2,
                "Spearman's Rho": round(spearman_corr, 2),
                "Spearman's p-value": round(spearman_p, 4),
                "Kendall's Tau": round(kendall_corr, 2),
                "Kendall's p-value": round(kendall_p, 4)
            })
correlation_results_df = pd.DataFrame(correlation_results)
correlation_results_df.to_csv('correlation_results_5min.csv', index=False)
correlation_results_df

In [ ]:
# Create heatmaps for Spearman's rho and Kendall's tau  
import plotly.express as px
# Pivot data for heatmap
spearman_pivot = correlation_results_df.pivot(index='Environmental Variable', columns='Environmental Variable2', 
                                              values="Spearman's Rho")
kendall_pivot = correlation_results_df.pivot(index='Environmental Variable', columns='Environmental Variable2', 
                                             values="Kendall's Tau")   
# Spearman's rho heatmap
fig_spearman = px.imshow(
    spearman_pivot,
    text_auto=".2f",
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    title="Spearman's Rho Correlation Heatmap"
)
fig_spearman.update_layout(
    title_font=dict(size=24, family='Serif'),
    font=dict(size=18, family='Serif')
)
fig_spearman.show()
fig_spearman.write_image("spearman_rho_heatmap_5min.pdf", format="pdf", width=800, height=600, scale=1)

In [ ]:
# Kendall's tau heatmap
fig_kendall = px.imshow(
    kendall_pivot,
    text_auto=".2f",
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    title="Kendall's Tau Correlation Heatmap"
)
fig_kendall.update_layout(
    title_font=dict(size=24, family='Serif'),
    font=dict(size=18, family='Serif')
)
fig_kendall.show()
fig_kendall.write_image("kendall_tau_heatmap_5min.pdf", format="pdf", width=800, height=600, scale=1)

### Computing correlation factors of continuos vars with behavioal vars using Spearman's rho and Kendall's tau

In [ ]:
# Computing correlation factors using Spearman's rho and Kendall's tau 
# with the subsampled dataset every 5 minutes
correlation_results = []
for env_var in (cleaned_ieq_vars):
    for beh in student_vars:
        df_pair = data_5min[[env_var, beh]]
        if df_pair.empty:
            continue
        # Spearman's rho
        spearman_corr, spearman_p = stats.spearmanr(df_pair[env_var], df_pair[beh])
        # Kendall's tau
        kendall_corr, kendall_p = stats.kendalltau(df_pair[env_var], df_pair[beh])
        # Store results
        correlation_results.append({
            "Environmental Variable": env_var,
            "Behavior": beh,
            "Spearman's Rho": spearman_corr,
            "Spearman's p-value": spearman_p,
            "Kendall's Tau": kendall_corr,
            "Kendall's p-value": kendall_p
        })  
# Convert results to DataFrame for better visualization
correlation_df = pd.DataFrame(correlation_results)
# Add pvalues to the dataframe
correlation_df = correlation_df[['Environmental Variable', 'Behavior', "Spearman's Rho", "Spearman's p-value", "Kendall's Tau", "Kendall's p-value"]]
# Display the correlation results
correlation_df.to_csv("correlation_results.csv", index=False)
correlation_df

### Cramer's V of the AQI versus Engagement, Attention, and Interaction

In [ ]:
# Cramér's V calculation between AQI and each student behavior variable with the whole dataset
def cramers_v(confusion_matrix):
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    # Standard formula: sqrt( chi2 / (n * (min(k, r) - 1)) )
    return np.sqrt(chi2 / (n * (min(k, r) - 1)))

cramers_results = []

for beh in student_vars:
    confusion_matrix = pd.crosstab(df_cat['AQI'], df_cat[beh])
    if confusion_matrix.empty:
        continue
    cramers_corr = cramers_v(confusion_matrix)
    chi2, p, dof, ex = stats.chi2_contingency(confusion_matrix)
    cramers_results.append({
        "Behavior": beh,
        "Cramer's V": cramers_corr,
        "p-value": p
    })
# Convert results to DataFrame for better visualization
cramers_df = pd.DataFrame(cramers_results)
# Display the Cramer's V results
cramers_df.to_csv("cramers_v_results_AQI.csv", index=False)
print(cramers_df)


In [ ]:
# Cramér's V calculation between AQI and each student behavior variable witht he subsampled dataset every 5 minutes

cramers_results = []

for beh in student_vars:
    confusion_matrix = pd.crosstab(df_cat_5min['AQI'], df_cat_5min[beh])
    if confusion_matrix.empty:
        continue
    cramers_corr = cramers_v(confusion_matrix)
    chi2, p, dof, ex = stats.chi2_contingency(confusion_matrix)
    cramers_results.append({
        "Behavior": beh,
        "Cramer's V": cramers_corr,
        "p-value": p
    })
# Convert results to DataFrame for better visualization
cramers_df = pd.DataFrame(cramers_results)
# Display the Cramer's V results
cramers_df.to_csv("cramers_v_results_AQI.csv", index=False)
print(cramers_df)


<a id=''></a>
## 5. Cross-Correlation Function analysis

In [ ]:
# Modified CCF analysis: use Spearman's rho per lag, annotate single peak, increase vertical spacing.
MAX_LAG_SAMPLES = 20  # ±20 samples -> ±100 minutes for 5-min sampling, for 1h40min lessons
FIG_PATH = "ccf_21_subplots_spearman_single_peak.pdf"  # save as PDF
CSV_PATH = "ccf_summary_spearman_single_peak.csv"
MIN_PER_SAMPLE = 5

# increase global font sizes for readability
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "figure.titlesize": 18
})

# ensure required objects exist
try:
    data_clean = data_5min.copy()
    ieq_list = ieq_vars
    beh_list = student_vars
except NameError:
    raise RuntimeError("Expected `data_cleaned`, `ieq_vars`, and `student_vars` to be defined in the notebook.")

if 'AQI' in ieq_list:
    ieq_list.remove('AQI')
if 'eCO2' in ieq_list:
    ieq_list.remove('eCO2')

In [ ]:
def cross_corr_symmetric_spearman(x, y, max_lag):
    """
    Compute Spearman rank correlation for lags -max_lag..+max_lag (symmetric).
    Uses overlapping-slice Spearman correlation (handles different effective N per lag).
    Returns array of length 2*max_lag+1.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    if x.size != y.size:
        raise ValueError("x and y must have same length for the window provided.")

    n = len(x)
    corr = np.full(2 * max_lag + 1, np.nan, dtype=float)
    idx = 0
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            lag_abs = abs(lag)
            x_sub = x[lag_abs:]
            y_sub = y[:-lag_abs]
        elif lag > 0:
            x_sub = x[:-lag]
            y_sub = y[lag:]
        else:
            x_sub = x
            y_sub = y

        if len(x_sub) < 3:
            corr[idx] = np.nan
        else:
            # use Spearman's rho on the overlapping slices
            rho, pval = stats.spearmanr(x_sub, y_sub)
            corr[idx] = float(rho) if not np.isnan(rho) else np.nan
        idx += 1
    return corr

In [ ]:
# Prepare figure: wider to allow increased horizontal spacing, keep vertical spacing increased
n_rows = len(ieq_list)
n_cols = len(beh_list)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 22), sharex=True)  # widened figsize
if n_rows == 1 and n_cols == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes[np.newaxis, :]
elif n_cols == 1:
    axes = axes[:, np.newaxis]

lags = np.arange(-MAX_LAG_SAMPLES, MAX_LAG_SAMPLES + 1)
lag_minutes = lags * MIN_PER_SAMPLE  # minutes per sample

summary_records = []

for i, ieq in enumerate(ieq_list):
    for j, beh in enumerate(beh_list):
        ax = axes[i, j]
        pair = data_clean[[ieq, beh]].dropna()
        n = len(pair)
        ax.set_title(f"{ieq} → {beh}", fontsize = 16)
        if n < 10:
            ax.text(0.5, 0.5, "Not enough data", ha="center", va="center")
            ax.set_xlim(lag_minutes.min(), lag_minutes.max())
            continue

        x = pair[ieq].values
        y = pair[beh].values
        ccf = cross_corr_symmetric_spearman(x, y, MAX_LAG_SAMPLES)

        # Confidence intervals per lag (approximate): using N_eff = n - |lag| and 99% of confidence, alpha = 0.01
        N_eff = n - np.abs(lags)
        with np.errstate(divide="ignore", invalid="ignore"):
            ci = 2.576 / np.sqrt(N_eff)
            ci[N_eff <= 0] = np.nan

        # ===== CCF curve =====
        ax.plot(
            lag_minutes,
            ccf,
            linewidth=1.3,
        )

        # CI bands
        ax.fill_between(lag_minutes, ci, -ci, color="gray", alpha=0.20)

        # Zero lines
        ax.axvline(0, color="black", linewidth=0.8)
        ax.axhline(0, color="gray", linestyle="--", linewidth=0.7)

        # Find single (global) peak in absolute correlation and annotate only that one
        valid_mask = ~np.isnan(ccf)
        annotated = []
        if np.any(valid_mask):
            # ignore NaNs for peak detection; use absolute ccf
            abs_ccf = np.abs(ccf.copy())
            abs_ccf[~valid_mask] = -np.inf

            # find local peaks; if none, fallback to global max index
            peak_idxs, _ = find_peaks(abs_ccf, distance=1)
            if peak_idxs.size == 0:
                global_pk = int(np.nanargmax(np.abs(ccf)))
            else:
                # choose the peak with largest absolute value among detected local peaks
                global_pk = int(peak_idxs[np.argmax(abs_ccf[peak_idxs])])
                # Check if the global peak value is lower than the extreme borders then make it the global max
                if (ccf[global_pk] > 0 and np.abs(ccf[global_pk]) < np.abs(ccf[len(ccf) - 1])) \
                    or (ccf[global_pk] < 0 and np.abs(ccf[global_pk]) < np.abs(ccf[0])):
                    global_pk = int(np.nanargmax(np.abs(ccf)))

            peak_lag = int(lags[global_pk])
            peak_val = float(ccf[global_pk])
            lag_m = int(lag_minutes[global_pk])

            # plot and annotate only the global peak
            ax.plot(lag_m, peak_val, "ro", markersize=9)
            y_off = 10 if peak_val >= 0 else -14
            ax.annotate(f"{lag_m}m\n{peak_val:.3f}", xy=(lag_m, peak_val),
                        xytext=(6, y_off), textcoords="offset points",
                        arrowprops=dict(arrowstyle="->", lw=0.9), fontsize=16)
            annotated.append((lag_m, float(peak_val)))
        else:
            peak_lag = None
            peak_val = None

        ax.set_xlim(lag_minutes.min(), lag_minutes.max())
        ax.set_ylim(-1.0, 1.0)
        ax.tick_params(axis='both', which='major', labelsize=16)
        if i == n_rows - 1:
            ax.set_xlabel("Lag (minutes)", fontsize=16)
        if j == 0:
            ax.set_ylabel("Spearman's rho", fontsize=16)

        # small label for the annotated single peak
        if annotated:
            ann_text = f"{annotated[0][0]}m:{annotated[0][1]:.2f}"
            ax.text(0.01, 0.95, ann_text, transform=ax.transAxes, fontsize=16,
                    verticalalignment='top', bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8))

        summary_records.append({
            "IEQ Variable": ieq,
            "Behavior": beh,
            "n_pairs": n,
            "peak_lag_bins": int(peak_lag) if peak_lag is not None else None,
            "peak_lag_minutes": int(peak_lag * MIN_PER_SAMPLE) if peak_lag is not None else None,
            "peak_correlation": float(peak_val) if peak_val is not None else None,
            "annotated_peaks": f"{annotated[0][0]}m:{annotated[0][1]:.3f}" if annotated else ""
        })

# Increase vertical spacing, increase horizontal spacing as requested
plt.tight_layout(rect=[0, 0, 1, 0.96], h_pad=0.7, w_pad=1.0)  # increased w_pad
# increase wspace here to provide more horizontal room between columns
fig.subplots_adjust(hspace=0.2, wspace=0.15, left=0.03, right=0.99, top=0.94, bottom=0.03)

# Save outputs (PDF)
fig.savefig(FIG_PATH, format="pdf", dpi=300, bbox_inches="tight")
pd.DataFrame(summary_records).to_csv(CSV_PATH, index=False)

print(f"Saved figure to {FIG_PATH}")
print(f"Saved summary to {CSV_PATH}")


### Recomputing prewhitened cross-correlation functions (Spearman's rho) with moving-block bootstrap confidence intervals.

**Parameters**
> MAX_LAG_SAMPLES = 20  # ±20 samples -> ±100 minutes for 5-min sampling, for 1h40min lessons
>
> max_ar_lag = 1       # as observed autocorr decay after 1 lag (in the 5-min resampled dataset) = 60 lags in the first dataset (sampled every 5 second)
>
> n_bootstrap = 1000
>
> block_length = 10
>
> random_seed = 42
>
> sampling_period_min = 5

In [ ]:
# ============================================
# Prewhitened CCFs for 15 IEQ–Behavior pairs
# With parallel bootstrap CIs (8 jobs)
# ============================================

timestamp_col = "Date"             # or None if no timestamp column

sampling_period_min = 5.0     # sampling interval (minutes)
max_lag = 20                  # lag in steps (20 → ±100 minutes for 5-min sampling, seems reasonable for 1h40min lessons)
max_ar_lag = 1               # AR order selection limit, as observed autocorr decays within ~60 lags = 5min
n_bootstrap = 2000            # bootstrap samples
block_length = 5             # moving-block bootstrap block length
n_jobs = 8                    # parallel bootstrap jobs

out_plot = "prewhitened_21_ccfs.png"
out_csv = "prewhitened_21_ccfs_summary.csv"

In [ ]:
# ============================================================
# FIT and PREWHITEN function
# ============================================================

def fit_ar_and_prewhiten(x, y, max_ar_lag=1):
    best_aic = np.inf
    best_model = None
    best_order = 0

    for lag in range(1, max_ar_lag + 1):
        try:
            m = AutoReg(x, lags=lag, old_names=False).fit()
            if m.aic < best_aic:
                best_aic = m.aic
                best_model = m
                best_order = lag
        except:
            pass

    if best_model is None:
        return np.asarray(x), np.asarray(y), 0, np.asarray([])

    p = best_order
    params = best_model.params
    ar_coefs = np.asarray(params[1:])  # skip intercept

    rx = np.asarray(best_model.resid)  # len = N - p
    y = np.asarray(y)
    N = len(y)

    # filter y with same AR model
    if p < 1:
        y_filt = y.copy()
    else:
        y_filt = np.zeros(N - p)
        for t in range(p, N):
            v = y[t]
            for i in range(1, p + 1):
                v -= ar_coefs[i - 1] * y[t - i]
            y_filt[t - p] = v

    L = min(len(rx), len(y_filt))
    return rx[-L:], y_filt[-L:], p, ar_coefs

In [ ]:
def spearman_ccf(rx, y_filt, max_lag):
    N = len(rx)
    lags = np.arange(-max_lag, max_lag + 1)
    rhos = np.full(len(lags), np.nan)

    for i, k in enumerate(lags):
        if k < 0:
            xr = rx[:N+k]
            yr = y_filt[-k:]
        elif k > 0:
            xr = rx[k:]
            yr = y_filt[:N-k]
        else:
            xr = rx
            yr = y_filt

        if len(xr) > 2:
            rhos[i] = spearmanr(xr, yr)[0]

    return lags, rhos

In [ ]:
def bootstrap_once(rx, y_filt, max_lag, block_length, seed):
    rng = np.random.default_rng(seed)
    N = len(rx)
    pairs = np.vstack([rx, y_filt]).T

    if block_length >= N:
        idx = rng.integers(0, N, size=N)
        boot_pairs = pairs[idx]
    else:
        n_blocks = int(np.ceil(N / block_length))
        starts = rng.integers(0, N - block_length + 1, size=n_blocks)
        blocks = [pairs[s:s+block_length] for s in starts]
        boot_pairs = np.vstack(blocks)[:N]

    rx_b = boot_pairs[:,0]
    y_b = boot_pairs[:,1]
    _, rhos_b = spearman_ccf(rx_b, y_b, max_lag)
    return rhos_b

In [ ]:
def bootstrap_parallel(rx, y_filt, max_lag, n_bs, block_length, n_jobs):
    seeds = np.random.SeedSequence().spawn(n_bs)
    replicates = Parallel(n_jobs=n_jobs)(
        delayed(bootstrap_once)(rx, y_filt, max_lag, block_length, s)
        for s in seeds
    )
    return np.vstack(replicates)

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

df = data_5min.copy()

print("Loaded data with columns:")
print(df.columns.tolist())


# ============================================================
# PROCESS ALL 21 PAIRS
# ============================================================

results = {}
summary = []

for x in ieq_list:
    if x in ieq_list:
        for y in student_vars:
            print(f"\n=== Processing {x} → {y} ===")

            xs = df[x].astype(float).values
            ys = df[y].astype(float).values
            mask = ~np.isnan(xs) & ~np.isnan(ys)
            xs, ys = xs[mask], ys[mask]

            rx, y_filt, ar_order, ar_coefs = fit_ar_and_prewhiten(xs, ys, max_ar_lag)
            lags, rhos = spearman_ccf(rx, y_filt, max_lag)

            # Bootstrap CIs (parallel)
            boot = bootstrap_parallel(rx, y_filt, max_lag, n_bootstrap, block_length, n_jobs)
            # ci for 99% CI
            # ci_low = np.nanpercentile(boot, 0.5, axis=0)
            # ci_up  = np.nanpercentile(boot, 99.5, axis=0)
            # ci for 95% CI
            ci_low = np.nanpercentile(boot, 2.5, axis=0)
            ci_up  = np.nanpercentile(boot, 97.5, axis=0)
            

            # peak (largest |rho|)
            peak_idx = np.nanargmax(np.abs(rhos))
            peak_rho = rhos[peak_idx]
            peak_lag_steps = lags[peak_idx]
            peak_lag_min = peak_lag_steps * sampling_period_min

            results[(x,y)] = dict(
                lags=lags,
                rhos=rhos,
                ci_low=ci_low,
                ci_up=ci_up,
                ar_order=ar_order,
                ar_coefs=ar_coefs,
                peak_idx=peak_idx,
                peak_rho=peak_rho,
                peak_lag_steps=peak_lag_steps,
                peak_lag_min=peak_lag_min
            )

            summary.append({
                "xvar": x,
                "yvar": y,
                "peak_rho": peak_rho,
                "peak_lag_steps": int(peak_lag_steps),
                "peak_lag_minutes": float(peak_lag_min),
                "ci_lower_at_peak": ci_low[peak_idx],
                "ci_upper_at_peak": ci_up[peak_idx],
                "ar_order": ar_order,
                "ar_coefs": " ".join([f"{c:.4f}" for c in ar_coefs]) if ar_order > 0 else ""
            })


# ============================================================
# SAVE SUMMARY CSV
# ============================================================

summary_df = pd.DataFrame(summary)
summary_df.to_csv(out_csv, index=False)
print(f"\nSaved summary CSV → {out_csv}")


In [ ]:

# ==================================================================
# 21-subplot PDF figure (7 × 3)
#   - Prewhitened CCF curve
#   - Light-gray bootstrap CI (99%)
#   - Peak |ρ| marker & annotation
# ==================================================================

pdf_output_path = "prewhitened_21_ccfs_95CI.pdf"

ieq_vars_plot = [var for var in ieq_vars if var not in ('AQI','eCO2')]
rows = len(ieq_vars_plot)
cols = len(student_vars)

fig, axes = plt.subplots(
    nrows=rows, 
    ncols=cols,
    figsize=(16, 16),
    constrained_layout=True
)

for i, x in enumerate(ieq_vars_plot):
    for j, y in enumerate(student_vars):

        ax = axes[i, j]
        res = results[(x, y)]

        ax.set_title(f"{x} → {y}", fontsize=16)

        # Convert lag steps → minutes
        lag_min = res['lags'] * sampling_period_min

        rhos = res["rhos"]
        ci_low = res['ci_low']       # <-- Should already be 0.5% percentile for 99% CI
        ci_up  = res['ci_up']        # <-- Should already be 99.5% percentile
        peak_idx = res['peak_idx']

        # ===== Prewhitened CCF curve =====
        ax.plot(
            lag_min,
            rhos,
            # marker="o",
            linewidth=1.3,
            # label="Prewhitened Spearman CCF"
        )

        # ===== Confidence Band (light gray) =====
        # ax.fill_between(lags * sampling_period_min, lower, upper, color='lightgray', alpha=0.7, label="Bootstrap 99% CI")
        ax.fill_between(
            lag_min,
            ci_low,
            ci_up,
            color="lightgray",
            alpha=0.7,
            # label="99% CI"
        )

        # ===== Zero reference =====
        ax.axhline(0, color="black", linewidth=0.7)

        # ===== Peak highlight =====
        if not np.isnan(rhos[peak_idx]):
            ax.plot(
                lag_min[peak_idx], 
                rhos[peak_idx],
                marker="s", 
                markersize=6, 
                color="red"
            )
            ax.annotate(
                f"ρ={rhos[peak_idx]:.3f}\nlag={int(lag_min[peak_idx])} min",
                xy=(lag_min[peak_idx], rhos[peak_idx]),
                xytext=(8, 6),
                textcoords="offset points",
                fontsize=14,
                color="red"
            )

        
        ax.set_xlim([lag_min.min(), lag_min.max()])    

        # ===== Titles & Labels =====
        # ax.set_title(f"{x} → {y}", fontsize=16)

        if j == 0:
            ax.set_ylabel("Spearman ρ", fontsize=16)

        if i == rows - 1:
            ax.set_xlabel("Lag (minutes)", fontsize=16)

        ax.grid(alpha=0.3)


# ==== Save to PDF ====
fig.savefig(pdf_output_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved 21-subplot CCF figure as PDF → {pdf_output_path}")


<a id='6'></a>
## 6. Time-Varying Granger Causality tests

In [ ]:
# ============================================
# Time-varying Granger Causality Analysis
# ============================================
'''
Computes the Time-varying Granger causality between IEQ vars and Student behavior vars in both senses using a 
rolling windows approach
Only the p-values from the F-test at the specified maxlag are recorded for each window
Results are stored in the DataFrame tv_gc_df with columns:
- time: the ending index of the window
- ieq_var: the IEQ variable name
- student_var: the Student behavior variable name
- p_ieq_to_student: p-value for the null hypothesis that IEQ does not Granger-cause Student behavior
- p_student_to_ieq: p-value for the null hypothesis that Student behavior does not Granger-cause IEQ
A summary DataFrame is also created with statistics for each IEQ-Student pair:
- ieq_var: the IEQ variable name
- student_var: the Student behavior variable name
- n_windows: number of valid windows tested
- min_p_ieq_to_student: minimum p-value observed for IEQ -> Student
- min_p_student_to_ieq: minimum p-value observed for Student -> IEQ
- frac_windows_sig_ieq_to_student_at_<threshold>: fraction of windows with p-value below threshold for IEQ -> Student
- frac_windows_sig_student_to_ieq_at_<threshold>: fraction of windows with p-value below threshold for Student -> IEQ
'''

# Parameters (you can override before running)
window_size = globals().get("window_size", 360) # covering 30 minutes at 5-sec sampling
maxlag = globals().get("maxlag", 60)            # 60 lags = 5 minutes at 5-sec sampling
step = globals().get("step", 60)               # 60 steps = 5 minutes at 5-sec sampling
thresholds = [0.01, 0.05]

# Ensure inputs exist
if 'data' not in globals():
    raise RuntimeError("data DataFrame not found in the notebook.")
if 'ieq_list' not in globals() or 'student_vars' not in globals():
    raise RuntimeError("ieq_list or student_vars not found in the notebook.")

n = len(data)
rows = len(ieq_list)
cols = len(student_vars)

results_records = []
summary_records = []

# progress over all pairs and windows
pairs = [(ieq, stu) for ieq in ieq_list for stu in student_vars]

for ieq, stu in tqdm(pairs, desc="Pairs"):
    window_pvals_ieq_to_stu = []
    window_pvals_stu_to_ieq = []
    window_times = []
    for start in range(0, n - window_size + 1, step):
        win = data.iloc[start:start + window_size]
        # extract numeric arrays
        x_stu = win[stu].values.astype(float)
        y_ieq = win[ieq].values.astype(float)
        time_idx = start + window_size - 1

        p_ieq_to_student = np.nan
        p_student_to_ieq = np.nan

        # skip windows with NaNs or constant series
        if (np.isnan(x_stu).any() or np.isnan(y_ieq).any() or
            np.allclose(x_stu, x_stu[0]) or np.allclose(y_ieq, y_ieq[0])):
            window_pvals_ieq_to_stu.append(np.nan)
            window_pvals_stu_to_ieq.append(np.nan)
            window_times.append(time_idx)
            results_records.append({
                "time": int(time_idx),
                "ieq_var": ieq,
                "student_var": stu,
                "p_ieq_to_student": np.nan,
                "p_student_to_ieq": np.nan
            })
            continue

        # suppress warnings from statsmodels in case of convergence/degenerate windows
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                # Test: does ieq -> student? (pass [student, ieq] so second column is potential cause)
                data_for_test = np.column_stack([x_stu, y_ieq])
                res = grangercausalitytests(data_for_test, maxlag=maxlag, verbose=False)
                # use p-value from the chosen maxlag F-test
                p_ieq_to_student = float(res[maxlag][0]['ssr_ftest'][1])
            except Exception:
                p_ieq_to_student = np.nan

            try:
                # Test: does student -> ieq? (pass [ieq, student])
                data_for_test_rev = np.column_stack([y_ieq, x_stu])
                res2 = grangercausalitytests(data_for_test_rev, maxlag=maxlag, verbose=False)
                p_student_to_ieq = float(res2[maxlag][0]['ssr_ftest'][1])
            except Exception:
                p_student_to_ieq = np.nan

        window_pvals_ieq_to_stu.append(p_ieq_to_student)
        window_pvals_stu_to_ieq.append(p_student_to_ieq)
        window_times.append(time_idx)

        results_records.append({
            "time": int(time_idx),
            "ieq_var": ieq,
            "student_var": stu,
            "p_ieq_to_student": p_ieq_to_student,
            "p_student_to_ieq": p_student_to_ieq
        })

    # summary for this pair
    arr_ieq = np.array(window_pvals_ieq_to_stu, dtype=float)
    arr_stu = np.array(window_pvals_stu_to_ieq, dtype=float)
    n_windows = int(np.sum(~np.isnan(arr_ieq) | ~np.isnan(arr_stu)))
    min_p_ieq = float(np.nanmin(arr_ieq)) if np.any(~np.isnan(arr_ieq)) else np.nan
    min_p_stu = float(np.nanmin(arr_stu)) if np.any(~np.isnan(arr_stu)) else np.nan
    frac_sig_ieq = float(np.nanmean(arr_ieq < thresholds[0])) if np.any(~np.isnan(arr_ieq)) else np.nan
    frac_sig_stu = float(np.nanmean(arr_stu < thresholds[0])) if np.any(~np.isnan(arr_stu)) else np.nan

    summary_records.append({
        "ieq_var": ieq,
        "student_var": stu,
        "n_windows": n_windows,
        "min_p_ieq_to_student": min_p_ieq,
        "min_p_student_to_ieq": min_p_stu,
        f"frac_windows_sig_ieq_to_student_at_{thresholds[0]}": frac_sig_ieq,
        f"frac_windows_sig_student_to_ieq_at_{thresholds[0]}": frac_sig_stu
    })

# assemble DataFrame
tv_gc_df = pd.DataFrame(results_records)

In [ ]:
# Plotting
use_datetime = ("Date" in data.columns) and tv_gc_df["time"].max() < len(data)
if use_datetime:
    # map time index to datetime (end index of each window)
    tv_gc_df = tv_gc_df.copy()
    tv_gc_df["time_dt"] = data["Date"].iloc[tv_gc_df["time"]].values
    x_col = "time_dt"
else:
    x_col = "time"

fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(18, 16),
                         sharex='col', sharey='row', constrained_layout=True)

# normalize axes array indexing
axes = np.array(axes).reshape(rows, cols)

# Define the proxy artist for the legend
orange_patch = mpatches.Patch(color='orange', alpha=0.9, label='Significant causality (p < 0.01)')
blue_line = plt.Line2D([0], [0], color='C0', lw=1.2, label='p-value IEQ_var → Student_var')
red_line = plt.Line2D([0], [0], color='red', linestyle=':', lw=0.9, label='α = 0.01')

for i, ieq in enumerate(ieq_vars):
    for j, stu in enumerate(student_vars):    
        ax = axes[i, j]
        subset = tv_gc_df[(tv_gc_df["ieq_var"] == ieq) & (tv_gc_df["student_var"] == stu)].sort_values(by=x_col)
        if subset.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center")
            ax.set_title(f"{ieq} — {stu}")
            continue

        x = subset[x_col].values
        p_ieq_to_stu = subset["p_ieq_to_student"].values
        # p_stu_to_ieq = subset["p_student_to_ieq"].values

        ax.plot(x, p_ieq_to_stu, label=f"{ieq} → {stu}", color="C0", linewidth=1.2)
        # ax.plot(x, p_stu_to_ieq, label=f"{stu} → {ieq}", color="C1", linewidth=1.2, linestyle="--")
        ax.axhline(thresholds[0], color="red", linestyle=":", linewidth=0.9, label=f"α={thresholds[0]}")
        # ax.axhline(thresholds[1], color="orange", linestyle=":", linewidth=0.8, label=f"α={thresholds[1]}")

        # shaded significance regions for α = thresholds[0]
        sig_mask = (p_ieq_to_stu < thresholds[0])
        # sig_mask_rev = (p_stu_to_ieq < thresholds[0])
        if np.any(sig_mask):
            # ax.fill_between(x, 0, 1, where=sig_mask, transform=ax.get_xaxis_transform(),
            #                 color="C1", alpha=0.88, interpolate=True)
            # Using step="mid" ensures the shading behaves like a bar chart (rectangular)
            # alpha=0.3 is usually better for visibility of the line itself
            ax.fill_between(x, 0, 1, where=sig_mask, transform=ax.get_xaxis_transform(),
                            color="orange", alpha=0.9, step="mid")
        
        ax.set_ylim(-0.01, 1.01)
        ax.set_title(f"{ieq} — {stu}", fontsize=14)
        if i == rows - 1:
            label_suffix = " (datetime)" if use_datetime else " (index)"
            ax.set_xlabel("Time" + label_suffix, fontsize=14)
        if j == 0:
            ax.set_ylabel("p-value", fontsize=14)
        # if i == 0 and j == 0:
        # ax.legend(fontsize=14, loc="upper right")
        # Apply the legend to the first subplot (or all if preferred)
        if i == 0 and j == 0:
            ax.legend(handles=[blue_line, red_line, orange_patch], 
                    fontsize=14, 
                    loc="upper right", 
                    frameon=True)

if use_datetime:
    for ax in fig.axes:
        for label in ax.get_xticklabels():
            label.set_rotation(25)

# fig.suptitle("Time-varying Granger Causality (p-values): IEQ ↔ Student Behaviors", fontsize=16)

out_pdf = "time_varying_granger_causality_allpairs_from_scratch.pdf"
fig.savefig(out_pdf, dpi=300, bbox_inches="tight")
plt.close(fig)

out_tv_csv = "time_varying_granger_causality_results_allpairs_from_scratch.csv"
out_summary_csv = "time_varying_granger_causality_summary_from_scratch.csv"
tv_gc_df.to_csv(out_tv_csv, index=False)
pd.DataFrame(summary_records).to_csv(out_summary_csv, index=False)

print(f"Saved combined TV-GC figure to: {out_pdf}")
print(f"Saved TV-GC detailed results to: {out_tv_csv}")
print(f"Saved TV-GC summary to: {out_summary_csv}")

In [ ]:
tv_gc_df.to_csv("time_varying_granger_causality_results.csv", index=False)

<a id='7'></a>
## 7. Independence analysis with categorized variables

### Chi squared analysis with binned variables

In [ ]:
# ============================================
# Chi-squared tests between each IEQ binned variable and each student behavior binned variable
# ============================================

if 'df_cat' not in globals() or 'df_cat_5min' not in globals():
    raise RuntimeError("df_binned not found. Run the binning cell (CELL INDEX: 50) first.")

results = []
for ieq in ieq_list:
    ieq_col = f"{ieq}_cat"
    for beh in student_vars:
        beh_col = f"{beh}_cat"

        sub = df_cat_5min[[ieq_col, beh_col]].dropna()
        if sub.empty:
            results.append({"IEQ": ieq, "Behavior": beh, "chi2": np.nan, "p_value": np.nan, "dof": np.nan, "n": 0})
            continue

        table = pd.crosstab(sub[ieq_col], sub[beh_col])
        if table.values.sum() == 0 or table.shape[0] < 2 or table.shape[1] < 2:
            # not enough categories to run chi2
            results.append({"IEQ": ieq, "Behavior": beh, "chi2": np.nan, "p_value": np.nan, "dof": np.nan, "n": int(sub.shape[0])})
            continue

        chi2, p, dof, expected = stats.chi2_contingency(table)
        results.append({"IEQ": ieq, "Behavior": beh, "chi2": float(chi2), "p_value": float(p), "dof": int(dof), "n": int(sub.shape[0])})

results_df = pd.DataFrame(results)
results_df.to_csv("chi2_ieq_behavior_results.csv", index=False)
print(results_df)

In [ ]:
# Method to compute Cramer's V
def cramers_v(chi2, n, dof):
    k = dof + 1  # número de categorías
    return np.sqrt(chi2 / (n * (k - 1)))

# Adding a column with Cramer's V
results_df['cramers_v'] = results_df.apply(lambda row: cramers_v(row['chi2'], row['n'], row['dof']), axis=1)

# Create a ranking by behavioral var
rankings = {}
for behavior in results_df['Behavior'].unique():
    subset = results_df[results_df['Behavior'] == behavior].sort_values(by='cramers_v', ascending=False)
    rankings[behavior] = subset[['IEQ','cramers_v']].reset_index(drop=True)

# Print results
print("Cramer's V for each pair IEQ-behavior:")
print(results_df[['IEQ','Behavior','chi2','cramers_v']])

print("\nRankings by behavioral var:")
for behavior, ranking in rankings.items():
    print(f"\n{behavior}:")
    print(ranking)


### Chi squared results between IEQ vars and behavioral vars per category

In [ ]:
def compute_chi2_pairs(df, ieq_list, beh_vars, dropna=True, save_residual_tables=False, out_dir="chi2_results"):
    results = []
    residual_tables = {}
    if save_residual_tables:
        os.makedirs(out_dir, exist_ok=True)

    for ieq in ieq_list:
        ieq_col = f"{ieq}_cat"
        for beh in beh_vars:
            beh_col = f"{beh}_cat"
            # build contingency table
            ct = pd.crosstab(df[ieq_col], df[beh_col], dropna=dropna)
            if ct.size == 0 or ct.values.sum() == 0:
                continue

            chi2, p, dof, expected = chi2_contingency(ct, correction=False)
            # standardized residuals: (observed - expected) / sqrt(expected)
            sr = (ct.values - expected) / np.sqrt(expected)
            sr_df = pd.DataFrame(sr, index=ct.index, columns=ct.columns)

            results.append({
                "ieq_var": ieq,
                "behavior_var": beh,
                "chi2": float(chi2),
                "p_value": float(p),
                "dof": int(dof),
                "n": int(ct.values.sum()),
                "max_abs_standardized_residual": float(np.abs(sr).max())
            })

            residual_tables[f"{ieq}__vs__{beh}"] = sr_df
            if save_residual_tables:
                ct.to_csv(os.path.join(out_dir, f"contingency__{ieq}__vs__{beh}.csv"))
                pd.DataFrame(expected, index=ct.index, columns=ct.columns).to_csv(
                    os.path.join(out_dir, f"expected__{ieq}__vs__{beh}.csv"))
                sr_df.to_csv(os.path.join(out_dir, f"std_residuals__{ieq}__vs__{beh}.csv"))

    results_df = pd.DataFrame(results)
    return results_df, residual_tables

results_df, residual_tables = compute_chi2_pairs(df_cat_5min, ieq_list, student_vars, save_residual_tables=True)
print(results_df)
# print(residual_tables)

In [ ]:
def chi_squared_by_category(data, col1, col2):
    """
    Computes the chi-squared contribution for each category in a contingency table.

    Args:
        data (pd.DataFrame): The input DataFrame.
        col1 (str): The name of the first categorical variable column.
        col2 (str): The name of the second categorical variable column.

    Returns:
        pd.DataFrame: A DataFrame with the chi-squared contribution for each cell.
        float: The total chi-squared statistic.
    """
    # 1. Create the contingency table of observed frequencies
    observed_table = pd.crosstab(data[col1], data[col2])
    # 2. Perform the chi-squared test to get expected frequencies and total statistic
    # chi2_contingency returns: chi2 statistic, p-value, degrees of freedom, expected frequencies
    chi2_stat, p_value, dof, expected_array = chi2_contingency(observed_table)
    # Convert expected frequencies array to a pandas DataFrame for better readability and alignment
    expected_table = pd.DataFrame(expected_array, index=observed_table.index, columns=observed_table.columns)
    # 3. Compute the chi-squared contribution for each cell
    # The formula is (Observed - Expected)^2 / Expected
    chi2_contributions = (observed_table - expected_table)**2 / expected_table
    return chi2_contributions, chi2_stat

In [ ]:
# Prepare column names
col_cat = [f"{col}_{cat}" for col in student_vars for cat in ['low', 'mid', 'high']]
row_cat = [f"{col}_{cat}" for col in ieq_list for cat in ['low', 'mid', 'high']]
full_chi2_by_category_df = pd.DataFrame(index=row_cat, columns=col_cat, dtype=float)

In [ ]:
# Create a new dataframe with the results concatenates for each IEQ var and student behavior var
for ieq in ieq_list:
    ieq_col = f'{ieq}_cat'
    for beh in student_vars:
        beh_col = f'{beh}_cat'
        chi2_contributions_df, total_chi2 = chi_squared_by_category(df_cat_5min, ieq_col, beh_col)
        # Cocatenate the values of the three by three dataframe into the full dataframe
        for r in chi2_contributions_df.index:
            for c in chi2_contributions_df.columns:
                r_col = f"{ieq}_{r}"
                c_col = f"{beh}_{c}"
                full_chi2_by_category_df.at[r_col, c_col] = chi2_contributions_df.at[r, c]

In [ ]:
# Save to Latex or show. Change save_latex = True to save as latex file
save_latex = False
if save_latex:
    full_chi2_by_category_df.to_latex('chi2_by_category_table.tex', float_format="%.2f", na_rep="0.00")
full_chi2_by_category_df